In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('ME228_Clean_Dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=True)  # ← fix here
df = df.sort_values('datetime').reset_index(drop=True)

print("Starting shape:", df.shape)

# ── LAG FEATURES ──
df['AQI_lag1']  = df['AQI'].shift(1)
df['AQI_lag2']  = df['AQI'].shift(2)
df['AQI_lag24'] = df['AQI'].shift(24)

df['pm2_5_lag1']  = df['pm2_5_ugm3'].shift(1)
df['pm2_5_lag24'] = df['pm2_5_ugm3'].shift(24)
df['o3_lag1']     = df['o3_ugm3'].shift(1)
df['o3_lag24']    = df['o3_ugm3'].shift(24)

# ── ROLLING WINDOW AVERAGES ──
df['AQI_roll3']  = df['AQI'].shift(1).rolling(window=3).mean()
df['AQI_roll6']  = df['AQI'].shift(1).rolling(window=6).mean()
df['AQI_roll24'] = df['AQI'].shift(1).rolling(window=24).mean()

df['pm2_5_roll3']  = df['pm2_5_ugm3'].shift(1).rolling(window=3).mean()
df['pm2_5_roll24'] = df['pm2_5_ugm3'].shift(1).rolling(window=24).mean()

# ── PEAK HOUR FLAG ──
df['is_peak_hour'] = df['hour'].apply(
    lambda x: 1 if (7 <= x <= 10) or (17 <= x <= 21) else 0
)

# ── DROP NaN ROWS ──
before = len(df)
df = df.dropna().reset_index(drop=True)
after = len(df)
print(f"Rows dropped due to lag NaN: {before - after}")
print(f"Final shape: {df.shape}")
print(f"Total features: {df.shape[1]}")

print("\nSample of new features:")
print(df[['datetime','AQI','AQI_lag1','AQI_lag24','AQI_roll3','AQI_roll24','is_peak_hour']].head(5))

# ── SAVE ──
df.to_csv('Featured_Dataset.csv', index=False)
print("\nSaved as Featured_Dataset.csv")

Starting shape: (29011, 34)
Rows dropped due to lag NaN: 24
Final shape: (28987, 47)
Total features: 47

Sample of new features:
             datetime  AQI  AQI_lag1  AQI_lag24  AQI_roll3  AQI_roll24  \
0 2022-08-07 05:00:00   47      51.0       46.0  48.666667   74.416667   
1 2022-08-07 06:00:00   42      47.0       40.0  49.000000   74.458333   
2 2022-08-07 07:00:00   36      42.0       41.0  46.666667   74.541667   
3 2022-08-07 08:00:00   39      36.0       51.0  41.666667   74.333333   
4 2022-08-07 09:00:00   46      39.0       60.0  39.000000   73.833333   

   is_peak_hour  
0             0  
1             0  
2             1  
3             1  
4             1  

Saved as Featured_Dataset.csv
